# CSE153R — Assignment 2: Symbolic Music Generation

| Section | Owner |
|---|---|
| Exploratory Analysis & Preprocessing | Nicholas |
| Modeling | Ulises |
| Evaluation | Josh |
| Related Work | Everyone |

**Tasks**
- **Task 1:** Symbolic Unconditioned Generation — LSTM Note Sequence Model (Nottingham Dataset)
- **Task 2:** Symbolic Conditioned Generation — Melody Harmonization via Seq2Seq LSTM (POP909)

## Shared Setup & Imports

In [ ]:
import os
import random
import numpy as np
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pretty_midi
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

---
# Task 1: Symbolic Unconditioned Generation
### LSTM Note Sequence Model — Nottingham Dataset

---
## 1.1 — Exploratory Analysis, Data Collection & Preprocessing
**Owner: Nicholas**

### Context
> **TODO (Nicholas):** Describe the Nottingham Dataset. Where does it come from? What is it for? How was it collected? (~2–3 paragraphs)
>
> Key points to cover:
> - Source: ~1,000 British and American folk tunes in ABC notation / MIDI format
> - Why this dataset: clean, monophonic melodies, well-studied benchmark, small enough to train quickly
> - How it was originally collected and by whom

### Data Processing Discussion
> **TODO (Nicholas):** Describe each processing step and justify the choices:
> - Parsing MIDI files with `pretty_midi`
> - Extracting `(pitch, duration_bucket)` pairs
> - Quantizing durations into 6 bins (sixteenth, eighth, quarter, dotted quarter, half, whole)
> - Building the vocabulary (~200–400 tokens expected)
> - Train/val/test 80/10/10 split by **file** (not by token) — explain why this prevents data leakage

In [ ]:
# TODO (Nicholas): Set path to the Nottingham MIDI dataset
NOTTINGHAM_DIR = 'data/nottingham/'  # update this path

# TODO (Nicholas): Load all MIDI files and verify count
# midi_files = [...]
# print(f'Total files: {len(midi_files)}')

In [ ]:
# TODO (Nicholas): Define duration quantization bins and vocabulary builder
# DURATION_BINS = [...]  # e.g. [0.125, 0.25, 0.5, 0.75, 1.0, 2.0] in beats

# def quantize_duration(dur):
#     """Map a raw duration (beats) to the nearest bin index."""
#     pass

# def parse_midi_to_tokens(midi_path):
#     """Return list of (pitch, dur_bin) tuples for one file."""
#     pass

# def build_vocabulary(all_sequences):
#     """Return token2id and id2token dicts. Include <PAD>, <EOS>, <START>."""
#     pass

In [ ]:
# TODO (Nicholas): Parse all files, build vocabulary, encode sequences
# all_sequences = [parse_midi_to_tokens(f) for f in midi_files]
# token2id, id2token = build_vocabulary(all_sequences)
# encoded_sequences = [[token2id[t] for t in seq] for seq in all_sequences]
# print(f'Vocabulary size: {len(token2id)}')

In [ ]:
# TODO (Nicholas): Train/val/test split by file (80/10/10)
# Shuffle file indices, then slice — do NOT split by token
# train_seqs, val_seqs, test_seqs = ...

### EDA — Plots & Statistics
> **TODO (Nicholas):** Produce the following plots and report key statistics in markdown cells below each plot.

In [ ]:
# TODO (Nicholas): Plot 1 — Vocabulary size report
# Print: total unique tokens, num special tokens, num pitch-only vs pitch+duration
pass

In [ ]:
# TODO (Nicholas): Plot 2 — Sequence length distribution (histogram)
# x-axis: number of tokens per file; report mean, median, max
pass

In [ ]:
# TODO (Nicholas): Plot 3 — Pitch histogram (MIDI pitch 0-127)
# Show which pitches appear most frequently across the corpus
pass

In [ ]:
# TODO (Nicholas): Plot 4 — Duration distribution (bar chart over 6 bins)
# Show relative frequency of each duration bucket
pass

In [ ]:
# TODO (Nicholas): Summary statistics table
# pd.DataFrame with: vocab_size, num_files, train/val/test sizes, avg_seq_len
pass

---
## 1.2 — Modeling
**Owner: Ulises**

### Context — Problem Formulation
> **TODO (Ulises):** Write 2–3 paragraphs answering:
> - How is this framed as an ML problem? Inputs, outputs, what is optimized?
> - Why is LSTM appropriate for this task vs. alternatives (Markov chain, Transformer)?
> - Key design table below — discuss each trade-off:

| Choice | Option A | Option B (chosen) | Why |
|---|---|---|---|
| Input representation | Raw MIDI pitch only | (pitch, duration) token | Captures rhythm, not just melody |
| Model | Markov chain | LSTM | Captures long-range dependencies |
| Sampling | Greedy | Temperature sampling | More diverse, musical output |
| Duration handling | Ignore | Quantized bins | Necessary for playable MIDI output |

### Architecture Discussion
> **TODO (Ulises):** Discuss advantages and disadvantages of:
> - Markov chains (fast, interpretable, no long-range memory)
> - LSTMs (long-range memory, more parameters, risk of overfitting on small data)
> - Transformers (powerful, overkill for ~1k short sequences, prone to overfitting)
> - Justify why LSTM is the right call for Nottingham's size and structure

In [ ]:
# TODO (Ulises): PyTorch Dataset — sliding window over encoded token sequences
# Window size: 128 tokens, stride: 64

class NottinghamDataset(Dataset):
    def __init__(self, sequences, seq_len=128, stride=64):
        # TODO: build (input, target) pairs using sliding windows
        # input  = tokens[i : i+seq_len]
        # target = tokens[i+1 : i+seq_len+1]  (shifted by 1)
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        pass

In [ ]:
# TODO (Ulises): LSTM Language Model
# Architecture: Embedding → LSTM (2 layers) → Linear → CrossEntropyLoss

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        # TODO: define self.embedding, self.lstm, self.fc, self.dropout
        pass

    def forward(self, x, hidden=None):
        # TODO: embed → lstm → linear → return logits and new hidden state
        pass

    def init_hidden(self, batch_size):
        # TODO: return zeroed (h_0, c_0) tensors on DEVICE
        pass

In [ ]:
# TODO (Ulises): Instantiate model, loss, optimizer, scheduler
# VOCAB_SIZE = len(token2id)  # from Nicholas's section
# model = LSTMLanguageModel(VOCAB_SIZE).to(DEVICE)
# criterion = nn.CrossEntropyLoss(ignore_index=token2id['<PAD>'])
# optimizer = optim.Adam(model.parameters(), lr=1e-3)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
# print(model)

In [ ]:
# TODO (Ulises): Training loop with early stopping
# - Log train loss and val loss per epoch
# - Early stopping patience = 5
# - Save best model checkpoint to 'task1_best_model.pt'

def train_epoch(model, loader, optimizer, criterion):
    # TODO
    pass

def evaluate(model, loader, criterion):
    # TODO
    pass

# TODO: training loop
# train_losses, val_losses = [], []
# for epoch in range(NUM_EPOCHS):
#     ...

In [ ]:
# TODO (Ulises): Plot training and validation loss curves
# x-axis: epoch; two lines: train loss, val loss
pass

In [ ]:
# TODO (Ulises): Temperature sampling / autoregressive generation
# Seed with <START> token, generate N tokens, stop on <EOS>
# Try temperatures: 0.8, 1.0, 1.2

def generate_melody(model, seed_tokens, id2token, temperature=1.0, max_len=200):
    # TODO: autoregressive loop — sample from softmax(logits / temperature)
    pass

In [ ]:
# TODO (Ulises): Decode generated token IDs → (pitch, duration) pairs → MIDI
# Save output as 'symbolic_unconditioned.mid'

def tokens_to_midi(token_sequence, id2token, output_path='symbolic_unconditioned.mid', tempo=120):
    # TODO: use pretty_midi to write notes from decoded (pitch, duration) pairs
    pass

# TODO: generate at T=0.8, 1.0, 1.2 and save the best one
# generated = generate_melody(model, [token2id['<START>']], id2token, temperature=1.0)
# tokens_to_midi(generated, id2token)

---
## 1.3 — Evaluation
**Owner: Josh**

### Context — Evaluation Framework
> **TODO (Josh):** Write 2–3 paragraphs covering:
> - What makes a "good" generated melody? (musicality, coherence, diversity)
> - How does perplexity relate to musical quality? What are its limits as a metric?
> - Why do we need multiple metrics (perplexity, pitch overlap, repetition ratio)?

### Baselines Discussion
> **TODO (Josh):** For each baseline, explain what it represents and why beating it matters:
> 1. **Unigram baseline** — always predict the most common token (dumb frequency prior)
> 2. **First-order Markov chain** — captures local transitions but no long-range structure
> 3. **Pitch-only LSTM (ablation)** — same architecture but ignores duration; shows value of rhythm encoding

In [ ]:
# TODO (Josh): Compute perplexity on the test set
# perplexity = exp(average cross-entropy loss over test tokens)

def compute_perplexity(model, loader, criterion):
    # TODO
    pass

In [ ]:
# TODO (Josh): Baseline 1 — Unigram model
# Always predicts the most frequent token in training set
# Compute its perplexity on the test set

def unigram_perplexity(train_seqs, test_seqs, vocab_size):
    # TODO
    pass

In [ ]:
# TODO (Josh): Baseline 2 — First-order Markov chain
# Build transition probability table from training sequences
# Compute perplexity on test set

def build_markov_chain(train_seqs, vocab_size):
    # TODO: return transition_matrix of shape (vocab_size, vocab_size)
    pass

def markov_perplexity(transition_matrix, test_seqs):
    # TODO
    pass

In [ ]:
# TODO (Josh): Baseline 3 — Pitch-only LSTM (ablation)
# Re-encode sequences using pitch only (drop duration), retrain a smaller LSTM
# Compare perplexity to full (pitch, duration) LSTM
# Note: coordinate with Ulises on model reuse
pass

In [ ]:
# TODO (Josh): Metric — Pitch class histogram overlap
# Compare pitch distribution of generated melodies vs. training corpus
# Overlap = sum(min(p_gen[c], p_train[c])) for c in 0..11

def pitch_histogram_overlap(generated_tokens, training_tokens, id2token):
    # TODO
    pass

In [ ]:
# TODO (Josh): Metric — Note duration distribution comparison (bar chart)
# Compare duration bin frequencies: generated vs. training
pass

In [ ]:
# TODO (Josh): Metric — Repetition ratio
# Fraction of generated n-grams (n=3) that appear more than once
# High ratio → model is looping/boring

def repetition_ratio(token_sequence, n=3):
    # TODO
    pass

In [ ]:
# TODO (Josh): Summary results table
# Rows: Unigram | Markov | Pitch-only LSTM | Full LSTM (ours)
# Columns: Perplexity | Pitch Overlap | Duration Match | Repetition Ratio

results_t1 = {
    'Model':            ['Unigram', 'Markov Chain', 'Pitch-only LSTM', 'Full LSTM (ours)'],
    'Perplexity':       [None, None, None, None],  # TODO: fill in
    'Pitch Overlap':    [None, None, None, None],
    'Duration Match':   [None, None, None, None],
    'Repetition Ratio': [None, None, None, None],
}
# pd.DataFrame(results_t1)

### Qualitative Evaluation
> **TODO (Josh):** Listen to 3–5 generated samples at T=0.8, 1.0, 1.2. Report which temperature sounds most musical and why. Embed/link MIDI files here.

---
## 1.4 — Related Work
**Owner: Everyone**

> **TODO (Everyone):** Write 3–5 paragraphs covering the three questions below. Aim for ~half a page.
>
> **How has this dataset / similar datasets been used before?**
> - Sturm et al. (2016) — "Music transcription modelling and composition using deep learning" — LSTMs on the same Nottingham dataset; reported perplexity ~4–6
> - (add 1–2 more references that used Nottingham or similar folk tune corpora)
>
> **How has prior work approached the same / similar tasks?**
> - Eck & Schmidhuber (2002) — early LSTM music generation on blues progressions
> - Boulanger-Lewandowski et al. (2012) — RNN-RBMs on polyphonic piano rolls (more complex extension)
>
> **How do your results compare?**
> - Compare your perplexity to Sturm et al.'s ~4–6 benchmark. Are you in the right ballpark? Discuss any differences.

---
# Task 2: Symbolic Conditioned Generation
### Melody Harmonization via Seq2Seq LSTM — POP909 Dataset

---
## 2.1 — Exploratory Analysis, Data Collection & Preprocessing
**Owner: Nicholas**

### Context
> **TODO (Nicholas):** Describe the POP909 dataset. (~2–3 paragraphs)
>
> Key points to cover:
> - 909 pop piano MIDIs, each with a pre-aligned chord annotation file
> - Three tracks per song: MELODY (Track 0), BRIDGE (Track 1), PIANO (Track 2)
> - `chord_midi.txt` format: `start_beat end_beat chord_label`
> - Why this dataset: well-studied, recognizable pop music, strong published baselines (REMI, PopMAG)

### Data Processing Discussion
> **TODO (Nicholas):** Describe and justify each step:
> - Loading melody from Track 0 only
> - Parsing and normalizing chord quality to 5 classes (major, minor, diminished, augmented, dominant7)
> - Duration quantization for melody tokens (same 6-bin scheme as Task 1)
> - Sliding window alignment: 8-note melody windows → majority chord label
> - Vocabulary sizes: melody (~200–300 tokens), chords (12 roots × 5 qualities × 6 dur bins = 360)
> - 80/10/10 split **by song** to prevent leakage

In [ ]:
# TODO (Nicholas): Set path to POP909 dataset (folders 001/ through 909/)
POP909_DIR = 'data/pop909/'  # update this path

# TODO (Nicholas): List all song directories and verify count
# song_dirs = sorted([os.path.join(POP909_DIR, d) for d in os.listdir(POP909_DIR)])
# print(f'Total songs: {len(song_dirs)}')

In [ ]:
# TODO (Nicholas): Chord quality normalizer
# Map 30+ raw quality strings → 5 classes: major, minor, diminished, augmented, dominant7

QUALITY_MAP = {
    # TODO: fill in mapping, e.g.
    # 'maj': 'major', 'maj7': 'major', 'maj6': 'major',
    # 'min': 'minor', 'min7': 'minor', ...
}

def normalize_quality(raw_quality):
    # TODO
    pass

def parse_chord_file(chord_path):
    """Return list of (start_beat, end_beat, root, quality, duration_beats)."""
    # TODO
    pass

In [ ]:
# TODO (Nicholas): Extract melody tokens from Track 0

def extract_melody_tokens(instrument, quantize_beats=0.25):
    """Return list of (pitch, dur_bin) tuples."""
    # TODO
    pass

def load_song(song_dir):
    """Return (melody_tokens, chord_annotations) for one POP909 song."""
    # TODO
    pass

In [ ]:
# TODO (Nicholas): Sliding window alignment
# Window: 8 melody notes, stride: 4
# For each window, find the majority chord during that time span
# Build list of (melody_window, chord_root, chord_quality, chord_duration_bin) tuples

def align_windows_to_chords(melody_tokens, chord_annotations, window=8, stride=4):
    # TODO
    pass

In [ ]:
# TODO (Nicholas): Build melody and chord vocabularies
# melody_vocab: (pitch, dur_bin) → id
# chord_vocab:  (root, quality, dur_bin) → id

# TODO (Nicholas): Run over all songs, collect windows, split 80/10/10 by song index
# train_windows, val_windows, test_windows = ...

### EDA — Plots & Statistics
> **TODO (Nicholas):** Produce the following plots.

In [ ]:
# TODO (Nicholas): Plot 1 — Chord label distribution histogram
# Is C major dominant? Show top-20 most common (root, quality) pairs
pass

In [ ]:
# TODO (Nicholas): Plot 2 — Melody pitch distribution vs. chord root correlation heatmap
# Rows: chord roots (C, D, E, ...); Columns: melody pitch classes
pass

In [ ]:
# TODO (Nicholas): Plot 3 — Average chord duration histogram
# Do pop songs favor 1-beat or 2-beat chords?
pass

In [ ]:
# TODO (Nicholas): Summary statistics table
# num_songs, train/val/test windows, melody_vocab_size, chord_vocab_size
pass

---
## 2.2 — Modeling
**Owner: Ulises**

### Context — Problem Formulation
> **TODO (Ulises):** Write 2–3 paragraphs covering:
> - Input: melody note window (sequence of (pitch, duration) tokens)
> - Output: sequence of (chord_root, chord_quality, chord_duration_bins)
> - What is being optimized: multi-task cross-entropy (chord label loss + 0.5 × duration loss)
> - Why seq2seq vs. a plain classifier? (models chord transition dependencies — IV→V→I is common)

### Architecture Discussion
> **TODO (Ulises):** Discuss trade-offs for each key design choice:

| Choice | Alternative | Why this choice |
|---|---|---|
| Bidirectional encoder | Unidirectional | Chord should "see" notes before AND after it |
| Bahdanau attention | Fixed context vector | Decoder can focus on harmonically relevant notes |
| Two output heads (label + duration) | Single combined token | Evaluate each independently; smaller per-head vocabulary |
| Window-based input | Full song sequence | Simpler batching; avoids long-sequence vanishing gradients |
| Quality normalization (5 classes) | Keep all 30+ types | Faster convergence, less data sparsity |

In [ ]:
# TODO (Ulises): PyTorch Dataset for (melody_window, chord_label, chord_duration) triples

class HarmonizationDataset(Dataset):
    def __init__(self, windows, melody_vocab, chord_vocab):
        # TODO: encode melody windows and chord targets as integer tensors
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        # TODO: return (melody_ids, chord_label_id, chord_dur_id)
        pass

In [ ]:
# TODO (Ulises): Bahdanau Attention module

class BahdanauAttention(nn.Module):
    def __init__(self, encoder_hidden_dim, decoder_hidden_dim):
        super().__init__()
        # TODO: define W_h, W_s, v linear layers
        # score(h_t, s_i) = v^T tanh(W_h * h_t + W_s * s_i)
        pass

    def forward(self, encoder_outputs, decoder_hidden):
        # TODO: return context vector and attention weights
        pass

In [ ]:
# TODO (Ulises): Seq2Seq Encoder
# Embedding → Bidirectional LSTM (2 layers, hidden=128, dropout=0.3)

class Encoder(nn.Module):
    def __init__(self, melody_vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        # TODO
        pass

    def forward(self, melody_ids):
        # TODO: return encoder_outputs (all hidden states), (h_n, c_n)
        # Note: bidirectional → concatenate forward+backward hidden states
        pass

In [ ]:
# TODO (Ulises): Seq2Seq Decoder with two output heads
# LSTM (hidden=256, 1 layer)
# Head 1 → chord label (root + quality combined) via cross-entropy
# Head 2 → chord duration bin via cross-entropy

class Decoder(nn.Module):
    def __init__(self, chord_label_vocab_size, chord_dur_vocab_size,
                 embed_dim=64, hidden_dim=256, encoder_hidden_dim=256):
        super().__init__()
        # TODO: chord embedding, LSTM, attention, head_label, head_duration
        pass

    def forward(self, prev_chord_id, decoder_hidden, encoder_outputs):
        # TODO: one decoder step
        # 1. embed previous chord
        # 2. compute attention context
        # 3. LSTM step with [embedding + context] as input
        # 4. project to chord label logits and duration logits
        # return label_logits, dur_logits, new_hidden
        pass

In [ ]:
# TODO (Ulises): Seq2Seq wrapper — wires Encoder + Attention + Decoder

class Seq2SeqHarmonizer(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        # TODO
        pass

    def forward(self, melody_ids, target_chord_ids=None, teacher_forcing_ratio=0.5):
        # TODO: encode melody; decode chord sequence autoregressively
        # Apply teacher forcing at rate teacher_forcing_ratio
        # Return stacked label logits and duration logits
        pass

In [ ]:
# TODO (Ulises): Instantiate model, losses, optimizer
# MELODY_VOCAB_SIZE = len(melody_vocab)  # from Nicholas
# CHORD_LABEL_VOCAB_SIZE = 12 * 5        # 12 roots × 5 qualities
# CHORD_DUR_VOCAB_SIZE = 6

# encoder = Encoder(MELODY_VOCAB_SIZE).to(DEVICE)
# decoder = Decoder(CHORD_LABEL_VOCAB_SIZE, CHORD_DUR_VOCAB_SIZE).to(DEVICE)
# model_t2 = Seq2SeqHarmonizer(encoder, decoder).to(DEVICE)

# criterion_label    = nn.CrossEntropyLoss()
# criterion_duration = nn.CrossEntropyLoss()
# optimizer_t2 = optim.Adam(model_t2.parameters(), lr=5e-4, weight_decay=1e-5)
# print(model_t2)

In [ ]:
# TODO (Ulises): Training loop for Task 2
# - total_loss = label_loss + 0.5 * duration_loss
# - teacher_forcing_ratio decays linearly from 0.5 → 0.0 over 15 epochs
# - early stopping on val chord accuracy (patience=5)
# - save best model to 'task2_best_model.pt'

def train_epoch_t2(model, loader, optimizer, criterion_label, criterion_duration, tf_ratio):
    # TODO
    pass

def evaluate_t2(model, loader, criterion_label, criterion_duration):
    # TODO: return avg loss, chord root accuracy, chord quality accuracy, duration accuracy
    pass

# TODO: training loop
# for epoch in range(NUM_EPOCHS_T2):
#     tf_ratio = max(0.0, 0.5 - epoch * (0.5 / 15))
#     ...

In [ ]:
# TODO (Ulises): Plot training/validation loss curves for Task 2
pass

In [ ]:
# TODO (Ulises): Generation — harmonize a held-out melody
# Use greedy decoding (optionally try beam search width=3)

def harmonize(model, melody_ids, chord_vocab, max_chords=32):
    """Autoregressively decode chord sequence for a melody window sequence."""
    # TODO
    pass

In [ ]:
# TODO (Ulises): Render harmonized output to MIDI
# Channel 0: original melody notes
# Channel 1: predicted block chords (root + third + fifth)
# Save as 'symbolic_conditioned.mid'

def harmonization_to_midi(melody_notes, predicted_chords, output_path='symbolic_conditioned.mid'):
    # TODO
    pass

# TODO: pick a test-set melody, run harmonize(), save MIDI
# predicted = harmonize(model_t2, test_melody_ids, chord_vocab)
# harmonization_to_midi(test_melody_notes, predicted)

---
## 2.3 — Evaluation
**Owner: Josh**

### Context — Evaluation Framework
> **TODO (Josh):** Write 2–3 paragraphs covering:
> - What makes a "good" harmonization? (correct chord roots, smooth transitions, consonance with melody)
> - Why is chord root accuracy not enough alone? (quality, harmonic rhythm, and consonance matter too)
> - The consonance score is grounded in music theory — explain the chord-tone concept

### Baselines Discussion
> **TODO (Josh):** Explain what each baseline captures and why the full model should outperform it:
> 1. **Most-common chord** (always predict C major) — pure frequency prior, lowest bar
> 2. **Key-following heuristic** — detect key signature, always predict tonic — stronger rule-based baseline
> 3. **Encoder-only MLP** — same encoder, no sequential decoder; shows value of modeling chord transitions

In [ ]:
# TODO (Josh): Metric — Chord root accuracy
# % of windows where predicted root matches ground-truth root

def chord_root_accuracy(predictions, targets):
    # TODO
    pass

In [ ]:
# TODO (Josh): Metric — Chord quality accuracy
# % of windows where predicted quality (major/minor/dim/aug/dom7) matches ground truth

def chord_quality_accuracy(predictions, targets):
    # TODO
    pass

In [ ]:
# TODO (Josh): Metric — Duration accuracy
# % of windows where predicted chord duration bin matches ground truth

def duration_accuracy(pred_durations, target_durations):
    # TODO
    pass

In [ ]:
# TODO (Josh): Metric — Chord transition perplexity
# Build a bigram model over ground-truth chord sequences in the test set
# Compute perplexity of predicted sequences under that bigram model

def chord_transition_perplexity(predicted_sequences, test_sequences):
    # TODO
    pass

In [ ]:
# TODO (Josh): Metric — Harmonic consonance score
# For each melody note, check if its pitch class is a chord tone of the predicted chord
# (root, third, fifth of the predicted (root, quality))
# Score = fraction of melody notes that ARE chord tones

CHORD_TONES = {
    # TODO: fill in intervals for each quality
    # e.g. 'major': [0, 4, 7], 'minor': [0, 3, 7], ...
}

def harmonic_consonance_score(melody_notes, predicted_chords):
    # TODO
    pass

In [ ]:
# TODO (Josh): Baseline 1 — Most-common chord (always predict C major)

def most_common_chord_baseline(test_windows, chord_vocab):
    # TODO: return predictions list (always the C major id)
    pass

In [ ]:
# TODO (Josh): Baseline 2 — Key-following heuristic
# Detect the most likely key from the melody pitch histogram
# Always predict the tonic chord of that key

def key_following_baseline(melody_tokens, chord_vocab):
    # TODO
    pass

In [ ]:
# TODO (Josh): Baseline 3 — Encoder-only MLP (ablation)
# Average encoder hidden states → MLP → predict chord label (no sequential decoder)
# Shows value of seq2seq decoder for capturing chord transitions

class EncoderOnlyMLP(nn.Module):
    def __init__(self, melody_vocab_size, chord_label_vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        # TODO
        pass

    def forward(self, melody_ids):
        # TODO: embed → avg pool → MLP → chord label logits
        pass

In [ ]:
# TODO (Josh): Summary results table for Task 2
# Rows: Most-common chord | Key heuristic | Encoder-only MLP | Seq2Seq (ours)
# Columns: Root Acc | Quality Acc | Duration Acc | Consonance Score | Transition PPL

results_t2 = {
    'Model':             ['Most-common chord', 'Key heuristic', 'Encoder-only MLP', 'Seq2Seq (ours)'],
    'Root Acc':          [None, None, None, None],  # TODO: fill in
    'Quality Acc':       [None, None, None, None],
    'Duration Acc':      [None, None, None, None],
    'Consonance Score':  [None, None, None, None],
    'Transition PPL':    [None, None, None, None],
}
# pd.DataFrame(results_t2)

### Qualitative Evaluation — Piano Roll Visualization
> **TODO (Josh):** For 1–2 test examples, plot a piano roll with chord label overlays.
> Color melody notes: **green** = chord tone, **red** = non-chord tone.
> Briefly comment on where the harmony sounds correct vs. where it clashes.

In [ ]:
# TODO (Josh): Piano roll + chord overlay visualization
# x-axis: time (beats); y-axis: MIDI pitch
# Shade chord regions; color each melody note green/red based on consonance

def plot_piano_roll_with_chords(melody_notes, predicted_chords, title='Harmonization'):
    # TODO
    pass

# TODO: call on 1–2 held-out examples

---
## 2.4 — Related Work
**Owner: Everyone**

> **TODO (Everyone):** Write 3–5 paragraphs covering the three questions below.
>
> **How has POP909 / similar datasets been used before?**
> - REMI (Huang & Yang 2020) — Transformer trained on POP909; same dataset, direct benchmark comparison
> - PopMAG (Ren et al. 2020) — multi-track pop music generation also using POP909
>
> **How has prior work approached the same / similar tasks?**
> - BachBot (Liang 2016) — LSTM harmonization of Bach chorales; comparable architecture
> - DeepBach (Hadjeres et al. 2017) — bidirectional LSTM; direct architectural comparison to your encoder
> - Harmonet (Hild et al. 1992) — classic neural harmonization baseline for historical context
>
> **How do your results compare?**
> - Compare chord accuracy to published numbers from DeepBach / BachBot (different domain but comparable task)
> - Note that REMI/PopMAG are more complex (full generation); your task is scoped to chord prediction only
> - Highlight the harmonic rhythm prediction as a novel contribution not present in most prior work

---
## Final Deliverables Checklist

| Item | Status |
|---|---|
| `symbolic_unconditioned.mid` | ☐ |
| `symbolic_conditioned.mid` | ☐ |
| Task 1: ≥3 baselines (unigram, Markov, pitch-only LSTM) | ☐ |
| Task 2: ≥3 baselines (most-common, key heuristic, MLP) | ☐ |
| Task 1: quantitative table (perplexity, pitch overlap, duration match, repetition ratio) | ☐ |
| Task 2: quantitative table (root acc, quality acc, duration acc, consonance score) | ☐ |
| Plots: pitch histogram, duration histogram, loss curves | ☐ |
| Task 2 piano roll visualization | ☐ |
| Related work sections (both tasks) | ☐ |